In [0]:
%sql
Select * from workspace.silver.silver_flights

In [0]:
dbutils.widgets.text("keycols","") # key columns
dbutils.widgets.text("cdccol","") # cdc columns
dbutils.widgets.text("backdated_refresh","") # backdated refresh
dbutils.widgets.text("source_object","") # source object
dbutils.widgets.text("source_schema","") # source schema

key_cols_list = eval(dbutils.widgets.get("keycols"))
cdc_col = dbutils.widgets.get("cdccol")
backdated_refresh = dbutils.widgets.get("backdated_refresh")
source_object = dbutils.widgets.get("source_object")
source_schema = dbutils.widgets.get("source_schema")

# since we will be implementing multithreading for processing, we wont we needing this cell. DONT RUN 

In [0]:
# catalog = "workspace"
# key_cols = "['flight_id']"
# key_cols_list = eval(key_cols)
# cdc_col = "modified_date"
# backdated_refresh = ""
# source_object = "silver_airports"
# source_schema = "silver"
# target_object = "dim_flights"
# target_schema = "gold"
# surrogate_key = "dim_flight_key"

In [0]:
# catalog = "workspace"
# key_cols = "['airport_id']"
# key_cols_list = eval(key_cols)
# cdc_col = "modified_date"
# backdated_refresh = ""
# source_object = "silver_airports"
# source_schema = "silver"
# target_object = "dim_airports"
# target_schema = "gold"
# surrogate_key = "dim_airport_key"

In [0]:
catalog = "workspace"
key_cols = "['passenger_id']"
key_cols_list = eval(key_cols)
cdc_col = "modified_date"
backdated_refresh = ""
source_object = "silver_passengers"
source_schema = "silver"
target_object = "dim_passengers"
target_schema = "gold"
surrogate_key = "dim_passenger_key"

In [0]:
# extracting last load date

# no backedated refresh
if len(backdated_refresh) == 0:

  # if table exists at destination
  if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
    last_load = spark.sql(f"select max({cdc_col}) from {catalog}.{target_schema}.{target_object}").collect()[0][0]
  else:
    last_load = "1900-01-01 00:00:00"

# perform backdated refresh
else :
  last_load = backdated_refresh
  
last_load

In [0]:
df_src = spark.sql(f"select * from {source_schema}.{source_object} where {cdc_col} > '{last_load}'")
df_src.display()

In [0]:


if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):

  # the columns already exists in the table, just need to select them
  key_cols_string_incremental = ", ".join(key_cols_list)
  df_trgt = spark.sql(f"select {key_cols_string_incremental}, {surrogate_key}, create_date, update_date from {catalog}.{target_schema}.{target_object}")

else:
  
  # the columns do not exists in the table, create them in the form - "" as A, "" as B 
  key_cols_string_initial = [f"'' as {i}" for i in key_cols_list]
  key_cols_string_initial = ", ".join(key_cols_string_initial)
  df_trgt = spark.sql(f"select {key_cols_string_initial}, 0 as {surrogate_key}, cast('1900-01-01 00:00:00' as timestamp) as create_date, cast('1900-01-01 00:00:00' as timestamp) as update_date where 1=0")

In [0]:
df_trgt.display()

In [0]:
join_condition = ' AND '.join([f"src.{i} = trgt.{i}" for i in key_cols_list])
join_condition

In [0]:
df_src.createOrReplaceTempView("src")
df_trgt.createOrReplaceTempView("trgt")

df_join = spark.sql(f"""
          select src.*,
          trgt.{surrogate_key},
          trgt.create_date,
          trgt.update_date
          from src
          left join trgt
          on {join_condition}
          """)

In [0]:
df_join.display()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

#old records 
df_old = df_join.filter(col(f'{surrogate_key}').isNotNull())

#new records
df_new = df_join.filter(col(f'{surrogate_key}').isNull())


In [0]:
# df_old will have nothing in case of initial load
df_old.display()
# df_new will have all records in case of initial load
df_new.display()


In [0]:
df_old_enriched = df_old.withColumn('update_date', current_timestamp())

In [0]:
# logic for handling the surrogate key.

if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
    max_surrogate_key = spark.sql(f"select max({surrogate_key}) from {catalog}.{target_schema}.{target_object}").collect()[0][0]
    df_new_enriched = df_new.withColumn(f'{surrogate_key}', lit(max_surrogate_key)+lit(1)+monotonically_increasing_id()) \
                    .withColumn('create_date', current_timestamp()) \
                    .withColumn('update_date', current_timestamp())
    
else:
    max_surrogate_key = 0
    df_new_enriched = df_new.withColumn(f'{surrogate_key}', lit(max_surrogate_key)+lit(1)+monotonically_increasing_id()) \
                            .withColumn('create_date', current_timestamp()) \
                            .withColumn('update_date', current_timestamp())
    
    

In [0]:
df_new_enriched.display()


In [0]:
df_union = df_old_enriched.union(df_new_enriched)
df_union.display()

In [0]:
# for upserts in delta tables
from delta.tables import * 

if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
    dlt_object = DeltaTable.forName(spark, f"{catalog}.{target_schema}.{target_object}")
    dlt_object.alias("trgt").merge(
        df_union.alias("src"),
        f"trgt.{surrogate_key} = src.{surrogate_key}"
    ).whenMatchedUpdateAll( condition = f"src.{cdc_col}=trgt.{cdc_col}") \
    .whenNotMatchedInsertAll() \
    .execute()
else:
    df_union.write.format("delta") \
            .mode("append") \
            .saveAsTable(f"{catalog}.{target_schema}.{target_object}")


In [0]:
%sql
select * from workspace.gold.dim_passengers